# Tentativo 18 · 1. Gli split sul punteggio

**L'obiettivo del tentativo.** Prevedere il **punteggio esatto NewsGuard (0–100)** di una testata dai suoi articoli,
senza classi in nessun punto: in addestramento e in valutazione conta solo il numero.

**Il problema degli split del tentativo 16.** I fold erano stratificati sui 5 livelli: ogni fold ha la stessa quota di
testate per livello, ma il valore **dentro** il livello è lasciato al caso (le testate l1 del fold 2 hanno in media 31
punti, quelle del fold 5 ne hanno 12). In più l'addestramento è per articolo e le testate hanno da 10 a 201 articoli: le
testate che scrivono molto decidono che cosa vede il modello.

**Qui.** Le testate si ordinano per punteggio e, a blocchi di 5 consecutive, ognuna va in un fold diverso: ogni fold ha
la stessa distribuzione del punteggio, anche dentro i livelli. Tre split con semi diversi, per sapere quanto i risultati
dipendono dallo split.

In [ ]:
T16_DIR = "../16_DataPreparation"
SEMI = "1,2,3"

In [ ]:
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

T16 = Path(T16_DIR).expanduser(); sys.path.insert(0, str(T16))
import funzioni as F16                      # il tentativo 16: dataset e cartella dei dati
import punteggio as P

warnings.filterwarnings("ignore")
plt.rcParams.update(F16.STILE_GRAFICI)
BLU, ARANCIO, GRIGIO, CHIARO, VERDE, ROSSO = "#2a78d6", "#eb6834", "#52514e", "#b5b3ab", "#1baf7a", "#b02a2a"
DATI18 = F16.DATI / "tentativi" / "18_PunteggioEsatto"; DATI18.mkdir(parents=True, exist_ok=True)
OUT = Path("Results"); OUT.mkdir(exist_ok=True)
art = pd.read_csv(F16.DATASET / "articoli.csv.gz")
t = pd.read_csv(F16.DATASET / "testate.csv", index_col="domain")
print(f"{len(art):,} articoli di {len(t)} testate")

In [ ]:
SEMI = [int(s) for s in str(SEMI).split(",")]
split = pd.DataFrame({f"split{s}": P.crea_split(t.score, s) for s in SEMI})
split.to_csv(DATI18 / "split.csv")
print(f"salvati in {DATI18 / 'split.csv'}")

from scipy.stats import ks_2samp
def equilibrio(fold):
    righe = []
    for k in range(1, 6):
        te, tr = t[fold == k], t[fold != k]
        righe.append({"fold": k, "punteggio medio": te.score.mean(), "media delle l1": te.score[te.label == 0].mean(),
                      "media delle l4": te.score[te.label == 3].mean(), "media per articolo": np.repeat(te.score, te.articoli).mean(),
                      "KS testate (p)": ks_2samp(te.score, tr.score).pvalue})
    return pd.DataFrame(righe).set_index("fold")
display(pd.concat({"split del tentativo 16 (per livello)": equilibrio(t.fold), "nuovo split 1 (per punteggio)": equilibrio(split.split1)}).round(2))

## Come sono distribuiti punteggi e articoli
A sinistra il punteggio delle 226 testate; al centro gli articoli per testata; a destra il punteggio delle testate di
ogni fold, nei vecchi split e nei nuovi.

In [ ]:
fig, assi = plt.subplots(1, 3, figsize=(16, 3.8))
assi[0].hist(t.score, bins=np.arange(0, 105, 5), color=BLU); assi[0].set_title("punteggio NewsGuard delle 226 testate"); assi[0].set_xlabel("punteggio")
assi[1].hist(t.articoli, bins=20, color=ARANCIO); assi[1].set_title("articoli per testata (da 10 a 201)"); assi[1].set_xlabel("articoli")
for dx, (nome, f) in zip([-0.18, 0.18], [("vecchi (per livello)", t.fold), ("nuovi (per punteggio)", split.split1)]):
    b = assi[2].boxplot([t.score[f == k] for k in range(1, 6)], positions=np.arange(1, 6) + dx, widths=0.3, patch_artist=True)
    for p_ in b["boxes"]: p_.set_facecolor(CHIARO if "vecchi" in nome else BLU)
assi[2].set_xticks(range(1, 6), [f"fold {k}" for k in range(1, 6)]); assi[2].set_title("punteggio per fold: vecchi (grigio) e nuovi (blu)")
plt.tight_layout(); plt.savefig(OUT / "split.png", dpi=170, bbox_inches="tight"); plt.show()